# TPC$_{RP}$ Active Learning — CIFAR-10

Implementation of the **TypiClust (TPC$_{RP}$)** algorithm from:
> Hacohen, G., Dekel, A., & Weinshall, D. (2022). *Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets.* ICML 2022.

**Strategy:** Self-supervised representation learning (simulated via pre-trained ResNet-18)  
→ K-Means clustering for diversity  
→ Typicality-based selection within each cluster.

In [ ]:
# ── Cell 1: Setup & Imports ──────────────────────────────────────────────────

# Mount Google Drive and add src/ to path when running on Colab
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/MachineLearning-Coursework2'
    sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
else:
    # Local: repo root is one level above notebooks/
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

# Standard library
import random
import warnings
warnings.filterwarnings('ignore')

# Numerical & ML
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# Clustering & neighbours
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

# Visualisation
import matplotlib
import matplotlib.pyplot as plt

# Project modules
from data_pipeline     import ActiveLearningDataset
from feature_extractor import ResNet18FeatureExtractor, extract_embeddings, build_extractor
from tpcrp_sampler     import tpcrp_query

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = (
    torch.device('cuda')  if torch.cuda.is_available() else
    torch.device('mps')   if torch.backends.mps.is_available() else
    torch.device('cpu')
)
print(f'Using device: {DEVICE}')
print(f'PyTorch {torch.__version__} | Torchvision {torchvision.__version__}')

In [ ]:
# ── Cell 2: Data Pipeline ────────────────────────────────────────────────────

DATA_ROOT = os.path.join(REPO_ROOT, 'data')

al_dataset = ActiveLearningDataset(root=DATA_ROOT, download=True)
print(al_dataset)
print(f'Classes : {al_dataset.classes}')
print(f'Train   : {len(al_dataset.full_dataset):,} examples')
print(f'Test    : {len(al_dataset.test_dataset):,} examples')

In [ ]:
# ── Cell 3: Feature Extraction ───────────────────────────────────────────────
#
# Extract L2-normalised 512-D embeddings from a pre-trained ResNet-18
# for the entire training set.  These are computed once and reused
# across all active learning iterations.

model, device = build_extractor(pretrained=True, device=DEVICE)

# Use the full (unlabeled) training set for embedding extraction.
# We create a temporary loader directly from the full dataset so that
# the index positions in all_embeddings align with the dataset indices.
full_loader = al_dataset.get_unlabeled_loader(batch_size=256, shuffle=False)

print('Extracting embeddings for the full training set...')
all_embeddings, all_labels = extract_embeddings(model, full_loader, device)

print(f'Embeddings shape : {all_embeddings.shape}')   # (50000, 512)
print(f'Labels shape     : {all_labels.shape}')       # (50000,)
print(f'Embedding norms  : min={np.linalg.norm(all_embeddings, axis=1).min():.4f}',
      f'max={np.linalg.norm(all_embeddings, axis=1).max():.4f}')  # should be ~1.0

In [ ]:
# ── Cell 4: TPC_RP Active Learning Loop (dry run) ───────────────────────────
#
# Ties together Dataset ▸ Feature Extractor ▸ TPC_RP Sampler.
# We run 5 iterations with a per-round budget of B=10 (one per class)
# and simply print the selected indices to verify the pipeline works.
# No downstream classifier is trained here.

N_ITERATIONS = 5
BUDGET       = 10   # B = number of classes in CIFAR-10

# Fresh start — all examples unlabeled
al_dataset.reset()

print('=' * 60)
print(f'TPC_RP dry run  |  {N_ITERATIONS} iterations  |  B={BUDGET} per round')
print('=' * 60)

for iteration in range(1, N_ITERATIONS + 1):

    # ── Run TPC_RP sampler ──────────────────────────────────────────────────
    queried_indices = tpcrp_query(
        all_features      = all_embeddings,
        unlabeled_indices = al_dataset.unlabeled_indices,
        labeled_indices   = al_dataset.labeled_indices,
        budget            = BUDGET,
        seed              = SEED + iteration,
    )

    # ── Move queried indices to the labeled pool ────────────────────────────
    al_dataset.query_samples(queried_indices)

    # ── Retrieve ground-truth classes for inspection ────────────────────────
    queried_classes = [al_dataset.full_dataset.targets[i] for i in queried_indices]
    class_names     = [al_dataset.classes[c] for c in queried_classes]

    # ── Report ──────────────────────────────────────────────────────────────
    cumulative_budget = al_dataset.num_labeled
    print(f'\nIteration {iteration}')
    print(f'  Queried indices  : {queried_indices}')
    print(f'  Classes selected : {class_names}')
    print(f'  Unique classes   : {len(set(queried_classes))} / {BUDGET}')
    print(f'  Cumulative budget: {cumulative_budget} labeled  |  '
          f'{al_dataset.num_unlabeled} unlabeled')

print('\n' + '=' * 60)
print('Dry run complete.  Pipeline verified.')
print(f'Total labeled after {N_ITERATIONS} rounds: {al_dataset.num_labeled}')